# 🧠 LENTERA Dataset Generation - Google Colab

Notebook ini akan generate **~1,520 training samples** untuk fine-tuning LENTERA AI.

## 📋 Checklist:
1. ✅ Set OpenAI API Key
2. ✅ Upload `golden_safety_responses.jsonl`
3. ✅ Run all cells
4. ✅ Download results

**Estimated Time**: 50-80 minutes  
**Estimated Cost**: ~$15-20 (OpenAI API)

---
## Step 1: Setup Environment

In [ ]:
# Install OpenAI library
!pip install -q openai
print("✅ OpenAI library installed")

---
## Step 2: Set API Key

⚠️ **IMPORTANT**: Masukkan OpenAI API key Anda di cell berikut

In [ ]:
import os

# 🔑 MASUKKAN API KEY ANDA DI SINI:
os.environ['OPENAI_API_KEY'] = "sk-proj-YOUR_KEY_HERE"  # 👈 GANTI INI!

# Verify
if os.environ['OPENAI_API_KEY'] == "sk-proj-YOUR_KEY_HERE":
    print("❌ ERROR: Please set your actual OpenAI API key!")
else:
    print("✅ API Key set successfully")

---
## Step 3: Upload Golden Responses File

📤 **Action Required**: 
1. Click the folder icon on the left sidebar
2. Upload file: `golden_safety_responses.jsonl` dari `C:\LenteraDreamFlow\Lenteraid\backend\`

In [ ]:
# Check if file uploaded
import os

if os.path.exists('golden_safety_responses.jsonl'):
    print("✅ Golden responses file found!")
    # Show preview
    !head -n 3 golden_safety_responses.jsonl
else:
    print("❌ File not found. Please upload golden_safety_responses.jsonl")

---
## Step 4: Dataset Generator Code

Core generation logic (sudah include semua improvements!).

In [ ]:
# Full generator code embedded
from openai import OpenAI
import json
import time
from typing import Dict, List, Optional
from collections import Counter

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
MODEL = "gpt-4o-mini"

print("✅ Generator initialized")

In [ ]:
# Load golden responses
def load_golden_responses():
    golden = []
    try:
        with open('golden_safety_responses.jsonl', 'r', encoding='utf-8') as f:
            for line in f:
                if line.strip():
                    golden.append(json.loads(line))
        print(f"✅ Loaded {len(golden)} golden responses")
        return golden
    except Exception as e:
        print(f"❌ Error loading golden responses: {e}")
        return []

golden_responses = load_golden_responses()

---
## Step 5: Generate Dataset

⏱️ **This will take ~50-80 minutes**

You'll see progress every 5-10 samples. Colab akan auto-save progress!

In [ ]:
# Quick test: Generate 5 samples first
print("🧪 Quick test: Generating 5 test samples...\n")

SYSTEM_PROMPT = '''You are an expert dataset generator for LENTERA mental health AI.

Generate realistic Indonesian mental health conversations following strict ethics:
- NO diagnosis, NO medication recommendations  
- Crisis responses MUST include hotlines (119 ext. 8)
- Natural Indonesian language (bisa campur English)
- Vary response openings (avoid repetitive patterns)

Output valid JSON only:
{
  "user_message": "...",
  "assistant_response": "...",
  "category": "emotional_validation",
  "risk_level": "low",
  "is_crisis": false,
  "crisis_confidence": "low"
}'''

def generate_sample(prompt_text):
    try:
        response = client.chat.completions.create(
            model=MODEL,
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": prompt_text}
            ],
            temperature=0.9,
            max_tokens=500
        )
        content = response.choices[0].message.content.strip()
        if content.startswith("```"):
            content = content.split("```")[1]
            if content.startswith("json"):
                content = content[4:]
            content = content.strip()
        return json.loads(content)
    except Exception as e:
        print(f"  ⚠️ Error: {e}")
        return None

# Test with 5 samples
test_samples = []
for i in range(5):
    sample = generate_sample(f"Generate a safe mental health conversation about stress. Variation {i+1}")
    if sample:
        test_samples.append(sample)
        print(f"✅ Sample {i+1}/5 generated")
    time.sleep(1)

print(f"\n✅ Test complete! Generated {len(test_samples)}/5 samples")
print("\nSample preview:")
print(json.dumps(test_samples[0], indent=2, ensure_ascii=False)[:300] + "...")

---
## ⚠️ DECISION POINT

**Test berhasil?** 
- ✅ YES → Lanjut ke full generation (1,520 samples)
- ❌ NO → Check API key / error message di atas

**After confirming test works, run cell below for FULL generation:**

---
## ⏱️ FULL GENERATION (Run this after test success)

**WARNING**: Ini akan:
- Generate ~1,520 samples
- Take 50-80 minutes
- Cost ~$15-20

Colab tab **MUST STAY OPEN** (tapi bisa minimize)

In [ ]:
# UNCOMMENT TO RUN FULL GENERATION:
# !wget https://raw.githubusercontent.com/YOUR_REPO/main/backend/finetuning/generate_enhanced_dataset.py
# !python generate_enhanced_dataset.py

# OR paste full code here:
print("⚠️ FULL GENERATION CODE")
print("Uncomment and run the wget command above, OR")
print("Upload generate_enhanced_dataset.py file and run: !python generate_enhanced_dataset.py")

---
## Alternative: Upload Script Directly

1. Upload `generate_enhanced_dataset.py` via file browser (left sidebar)
2. Run cell below:

In [ ]:
# Check if script exists
if os.path.exists('generate_enhanced_dataset.py'):
    print("✅ Script found! Running generation...\n")
    !python generate_enhanced_dataset.py
else:
    print("❌ Please upload generate_enhanced_dataset.py from your local machine")
    print("   Location: C:\\LenteraDreamFlow\\backend\\finetuning\\generate_enhanced_dataset.py")

---
## Step 6: Download Results

After generation complete, download files:

In [ ]:
# Check generated files
!ls -lh *.json 2>/dev/null || echo "No JSON files found yet"

# Download links (will appear after generation)
from google.colab import files

if os.path.exists('dataset_lentera_enhanced.json'):
    print("\n📥 Downloading dataset_lentera_enhanced.json...")
    files.download('dataset_lentera_enhanced.json')
    print("✅ Download started!")
else:
    print("⏳ Dataset not ready yet. Run generation first!")

---
## Step 7: Convert to Training Format

Convert to ShareGPT format untuk Axolotl training:

In [ ]:
# Simple converter (inline)
import json
import random

if os.path.exists('dataset_lentera_enhanced.json'):
    print("Converting to ShareGPT format...\n")
    
    with open('dataset_lentera_enhanced.json', 'r', encoding='utf-8') as f:
        dataset = json.load(f)
    
    # Convert
    sharegpt_data = []
    for sample in dataset:
        sharegpt_data.append({
            "conversations": [
                {"from": "human", "value": sample["user_message"]},
                {"from": "gpt", "value": sample["assistant_response"]}
            ],
            "category": sample.get("category"),
            "risk_level": sample.get("risk_level"),
            "is_crisis": sample.get("is_crisis")
        })
    
    # Shuffle and split
    random.shuffle(sharegpt_data)
    val_size = int(len(sharegpt_data) * 0.1)
    train_data = sharegpt_data[val_size:]
    val_data = sharegpt_data[:val_size]
    
    # Save
    with open('train.jsonl', 'w', encoding='utf-8') as f:
        for item in train_data:
            f.write(json.dumps(item, ensure_ascii=False) + '\n')
    
    with open('val.jsonl', 'w', encoding='utf-8') as f:
        for item in val_data:
            f.write(json.dumps(item, ensure_ascii=False) + '\n')
    
    print(f"✅ Conversion complete!")
    print(f"   Train: {len(train_data)} samples")
    print(f"   Val: {len(val_data)} samples")
    
    # Download
    print("\n📥 Downloading train.jsonl...")
    files.download('train.jsonl')
    print("📥 Downloading val.jsonl...")
    files.download('val.jsonl')
else:
    print("❌ Dataset file not found. Generate first!")

---
## ✅ DONE!

You should now have:
- ✅ `dataset_lentera_enhanced.json` (raw dataset)
- ✅ `train.jsonl` (~1,370 samples)
- ✅ `val.jsonl` (~150 samples)

**Next Step**: Use `train.jsonl` and `val.jsonl` for fine-tuning di Phase 2!